# Small Business Profitability Analysis — Animated Dashboard
**Google Colab | Python | Kaggle dataset**

This notebook explores sales, costs/profit, product categories, regions, discounts and monthly trends. It includes animated Plotly charts and an interactive dashboard.

> **Dataset note:** This project uses the Kaggle *Superstore Dataset – Final* as a retail small-business proxy. It contains sales and profit fields, but it is not a survey of independently owned small businesses. Interpret the results as retail transaction profitability.

## 1. Install dependencies
Run this cell first. Colab usually has pandas already; the remaining packages enable Kaggle download, Plotly animation, and dashboard widgets.

In [ ]:
%pip -q install kagglehub plotly ipywidgets

## 2. Download the dataset from Kaggle
The dataset is publicly listed on Kaggle. `kagglehub` downloads it without needing to upload a CSV manually.

Dataset: [Superstore Dataset – Final](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

If Kaggle download fails in your Colab session, download the CSV from the Kaggle page and use the fallback upload cell below.

In [ ]:
import kagglehub
from pathlib import Path
import pandas as pd
import glob, os

dataset_dir = Path(kagglehub.dataset_download("vivek468/superstore-dataset-final"))
csv_files = list(dataset_dir.rglob("*.csv"))
print("Downloaded files:")
for f in csv_files:
    print("-", f.name)

if not csv_files:
    raise FileNotFoundError("No CSV found in downloaded Kaggle dataset.")
# Pick the likely main dataset; if there are multiple CSVs, inspect the printed list.
csv_path = next((f for f in csv_files if "superstore" in f.name.lower()), csv_files[0])
df = pd.read_csv(csv_path, encoding="latin-1")
print("Loaded:", csv_path.name, "| shape:", df.shape)
display(df.head())

### Fallback: upload a CSV manually (only if Kaggle download failed)
Run this cell instead of the download/load cell above. It will ask you to select the dataset CSV.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# csv_path = next(iter(uploaded))
# df = pd.read_csv(csv_path, encoding="latin-1")
# display(df.head())

## 3. Clean and standardize column names
The dataset sometimes uses spaces and capitalization differences. This step maps common names to a consistent format.

In [ ]:
import numpy as np

df.columns = [str(c).strip() for c in df.columns]
# Normalize common variants while preserving useful original labels
rename_map = {}
for c in df.columns:
    key = c.strip().lower().replace("_", " ")
    if key in ["order date", "orderdate"]: rename_map[c] = "Order Date"
    elif key == "sales": rename_map[c] = "Sales"
    elif key == "profit": rename_map[c] = "Profit"
    elif key == "discount": rename_map[c] = "Discount"
    elif key == "quantity": rename_map[c] = "Quantity"
    elif key in ["category", "sub category", "subcategory"]: rename_map[c] = "Category" if key == "category" else "Sub-Category"
    elif key == "region": rename_map[c] = "Region"
    elif key == "segment": rename_map[c] = "Segment"
    elif key in ["product name", "product"]: rename_map[c] = "Product Name"
    elif key in ["order id", "orderid"]: rename_map[c] = "Order ID"

df = df.rename(columns=rename_map)
required = ["Sales", "Profit"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}. Available columns: {list(df.columns)}")

for c in ["Sales", "Profit", "Discount", "Quantity"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "Order Date" in df.columns:
    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
    df["Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
    df["Year"] = df["Order Date"].dt.year

df = df.dropna(subset=["Sales", "Profit"]).copy()
df["Profit Margin (%)"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"] * 100, np.nan)
df["Profit Status"] = np.where(df["Profit"] >= 0, "Profitable", "Loss")
print(f"Clean rows: {len(df):,}")
print("Columns:", list(df.columns))
display(df.head())

## 4. Executive KPI overview

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
margin = (total_profit / total_sales * 100) if total_sales else np.nan
orders = df["Order ID"].nunique() if "Order ID" in df.columns else len(df)

print(f"Total Sales:      {total_sales:,.2f}")
print(f"Total Profit:     {total_profit:,.2f}")
print(f"Profit Margin:    {margin:.2f}%")
print(f"Orders/Records:   {orders:,}")
print(f"Loss-making rows: {(df['Profit'] < 0).sum():,} ({(df['Profit'] < 0).mean()*100:.1f}%)")

## 5. Animated chart — monthly sales and profit
Plotly animation reveals how the monthly trend develops over time. The chart animates month-by-month.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

if "Month" in df.columns and df["Month"].notna().any():

    # 1. Monthly data
    monthly = (
        df.dropna(subset=["Month"])
        .groupby("Month", as_index=False)[["Sales", "Profit"]]
        .sum()
        .sort_values("Month")
    )

    monthly["Month Label"] = monthly["Month"].dt.strftime("%Y-%m")

    # 2. Create animation frames
    frames = []

    for i in range(len(monthly)):

        current = monthly.iloc[:i + 1]

        frames.append(
            go.Frame(
                name=str(i),
                data=[
                    go.Bar(
                        x=current["Month Label"],
                        y=current["Sales"],
                        name="Sales",
                        marker_color="#2563EB",
                        text=current["Sales"].apply(lambda x: f"{x:,.0f}"),
                        textposition="outside"
                    ),

                    go.Bar(
                        x=current["Month Label"],
                        y=current["Profit"],
                        name="Profit",
                        marker_color=[
                            "#00804A" if p >= 0 else "#C1122F"
                            for p in current["Profit"]
                        ],
                        text=current["Profit"].apply(lambda x: f"{x:,.0f}"),
                        textposition="outside"
                    )
                ]
            )
        )

    # 3. Initial chart
    fig = go.Figure(
        data=frames[0].data,
        frames=frames
    )

    # 4. Layout
    fig.update_layout(
        title=dict(
            text="Animated Monthly Sales & Profit",
            x=0.08,
            font=dict(size=22)
        ),

        template="plotly_white",

        xaxis_title="Month",
        yaxis_title="Amount",

        barmode="group",

        height=600,

        font=dict(
            family="Arial",
            size=14,
            color="#344054"
        ),

        plot_bgcolor="white",
        paper_bgcolor="white",

        # Animation buttons
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                x=0.1,
                y=1.15,

                buttons=[
                    dict(
                        label="Start Animation",
                        method="animate",
                        args=[
                            None,
                            {
                                "frame": {
                                    "duration": 800,
                                    "redraw": True
                                },
                                "transition": {
                                    "duration": 400
                                },
                                "fromcurrent": True,
                                "mode": "immediate"
                            }
                        ]
                    ),

                    dict(
                        label="Stop Animation",
                        method="animate",
                        args=[
                            [None],
                            {
                                "frame": {
                                    "duration": 0,
                                    "redraw": False
                                },
                                "mode": "immediate"
                            }
                        ]
                    )
                ]
            )
        ],

        # Month slider
        sliders=[
            dict(
                active=0,
                currentvalue={
                    "prefix": "Current Month: "
                },

                steps=[
                    dict(
                        label=monthly["Month Label"].iloc[i],
                        method="animate",
                        args=[
                            [str(i)],
                            {
                                "frame": {
                                    "duration": 500,
                                    "redraw": True
                                },
                                "mode": "immediate"
                            }
                        ]
                    )
                    for i in range(len(monthly))
                ]
            )
        ]
    )

    # 5. Axis styling
    fig.update_yaxes(
        gridcolor="#E5EAF0",
        zeroline=True,
        zerolinecolor="#B0B8C4"
    )

    fig.update_xaxes(
        showgrid=False
    )

    fig.show()

else:
    print("Order Date is unavailable; monthly animation skipped.")

## 6. Animated chart — category profit contribution
This animated chart compares category profit across time. If the dataset does not contain a category field, the cell will skip gracefully.

In [ ]:
if "Category" in df.columns and "Month" in df.columns and df["Month"].notna().any():
    cat_month = (df.dropna(subset=["Month"])
                   .groupby(["Month", "Category"], as_index=False)["Profit"].sum())
    cat_month["Month Label"] = cat_month["Month"].dt.strftime("%Y-%m")
    fig = px.bar(cat_month, x="Category", y="Profit", color="Category",
                 animation_frame="Month Label", title="Animated Profit by Category",
                 range_y=[min(0, cat_month["Profit"].min()*1.1), cat_month["Profit"].max()*1.1])
    fig.update_layout(template="plotly_white", showlegend=False)
    fig.show()
else:
    print("Category or Month field unavailable; category animation skipped.")

## 7. Interactive dashboard
Use the dropdown filters to explore category, region and customer segment. Charts and KPI totals update when you change the selections.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def options_for(col):
    if col not in df.columns:
        return ["All"]
    vals = sorted(df[col].dropna().astype(str).unique().tolist())
    return ["All"] + vals

category_filter = widgets.Dropdown(options=options_for("Category"), value="All", description="Category:")
region_filter = widgets.Dropdown(options=options_for("Region"), value="All", description="Region:")
segment_filter = widgets.Dropdown(options=options_for("Segment"), value="All", description="Segment:")
out = widgets.Output()

def filtered_data():
    data = df.copy()
    for col, val in [("Category", category_filter.value),
                     ("Region", region_filter.value),
                     ("Segment", segment_filter.value)]:
        if val != "All" and col in data.columns:
            data = data[data[col].astype(str) == val]
    return data

def render_dashboard(*args):
    data = filtered_data()
    with out:
        clear_output(wait=True)
        sales = data["Sales"].sum()
        profit = data["Profit"].sum()
        pm = profit / sales * 100 if sales else 0
        loss_rows = int((data["Profit"] < 0).sum())
        print(f"SALES: {sales:,.2f}    PROFIT: {profit:,.2f}    PROFIT MARGIN: {pm:.2f}%    LOSS ROWS: {loss_rows:,}")

        if "Category" in data.columns:
            cat = data.groupby("Category", as_index=False)[["Sales", "Profit"]].sum()
            fig1 = px.bar(cat, x="Category", y=["Sales", "Profit"], barmode="group",
                          title="Sales & Profit by Category", template="plotly_white")
            fig1.show()

        if "Region" in data.columns:
            reg = data.groupby("Region", as_index=False)["Profit"].sum().sort_values("Profit")
            fig2 = px.bar(reg, x="Profit", y="Region", orientation="h", color="Profit",
                          title="Profit by Region", template="plotly_white",
                          color_continuous_scale="RdYlGn")
            fig2.show()

        if "Discount" in data.columns:
            fig3 = px.scatter(data, x="Discount", y="Profit", color="Category" if "Category" in data.columns else None,
                              size="Sales", hover_data=[c for c in ["Product Name", "Region"] if c in data.columns],
                              title="Discount vs Profit", template="plotly_white")
            fig3.show()

        if "Month" in data.columns and data["Month"].notna().any():
            trend = data.dropna(subset=["Month"]).groupby("Month", as_index=False)[["Sales", "Profit"]].sum()
            fig4 = go.Figure()
            fig4.add_trace(go.Scatter(x=trend["Month"], y=trend["Sales"], mode="lines+markers", name="Sales"))
            fig4.add_trace(go.Scatter(x=trend["Month"], y=trend["Profit"], mode="lines+markers", name="Profit"))
            fig4.update_layout(title="Monthly Sales & Profit", template="plotly_white")
            fig4.show()

for w in [category_filter, region_filter, segment_filter]:
    w.observe(render_dashboard, names="value")

display(widgets.HBox([category_filter, region_filter, segment_filter]))
display(out)
render_dashboard()

## 8. Profitability deep dive
### Best and weakest categories / sub-categories

In [ ]:
for col in ["Category", "Sub-Category"]:
    if col in df.columns:
        summary = (df.groupby(col, as_index=False)
                     .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))
        summary["Profit Margin (%)"] = np.where(summary["Sales"] != 0, summary["Profit"] / summary["Sales"] * 100, np.nan)
        print(f"Profitability by {col}:")
        display(summary.sort_values("Profit", ascending=False))

### Discount bands and profit
Discounts are analyzed as observational associations—not proof that discounts alone cause losses.

In [ ]:
if "Discount" in df.columns:
    df["Discount Band"] = pd.cut(df["Discount"], bins=[-0.001, 0.05, 0.15, 0.25, 0.5, 1.0],
                                 labels=["0–5%", "5–15%", "15–25%", "25–50%", "50%+"])
    discount_summary = df.groupby("Discount Band", observed=False, as_index=False).agg(
        Sales=("Sales", "sum"), Profit=("Profit", "sum"), Avg_Discount=("Discount", "mean"))
    discount_summary["Profit Margin (%)"] = np.where(discount_summary["Sales"] != 0,
                                                      discount_summary["Profit"]/discount_summary["Sales"]*100, np.nan)
    display(discount_summary)
    fig = px.bar(discount_summary, x="Discount Band", y="Profit", color="Profit",
                 title="Profit by Discount Band", template="plotly_white",
                 color_continuous_scale="RdYlGn")
    fig.show()

## 9. Auto-generated business insights
These are calculated from the currently loaded dataset, not hard-coded findings.

In [ ]:
insights = []
if "Category" in df.columns:
    by_cat = df.groupby("Category")["Profit"].sum().sort_values(ascending=False)
    if len(by_cat):
        insights.append(f"Most profitable category: {by_cat.index[0]} ({by_cat.iloc[0]:,.2f} profit).")
        insights.append(f"Lowest-profit category: {by_cat.index[-1]} ({by_cat.iloc[-1]:,.2f} profit).")
if "Region" in df.columns:
    by_region = df.groupby("Region")["Profit"].sum().sort_values(ascending=False)
    if len(by_region):
        insights.append(f"Top profit region: {by_region.index[0]} ({by_region.iloc[0]:,.2f}).")
if "Discount" in df.columns:
    corr = df[["Discount", "Profit"]].corr().iloc[0, 1]
    insights.append(f"Correlation between discount and profit: {corr:.3f} (correlation is not causation).")
insights.append(f"Overall profit margin: {margin:.2f}%.")
for item in insights:
    print("•", item)

## 10. Export cleaned data and summaries
You can download the cleaned data and use it in Excel or Power BI.

In [ ]:
from google.colab import files

df.to_csv("cleaned_small_business_profitability.csv", index=False)
files.download("cleaned_small_business_profitability.csv")

In [ ]:
import pandas as pd
import numpy as np
import json
from IPython.display import HTML, display
from google.colab import files

# ==========================================
# 1. PREPARE DATA
# ==========================================

data = df.copy()

# Month-wise analysis
monthly = (
    data.dropna(subset=["Month"])
    .groupby("Month", as_index=False)[["Sales", "Profit"]]
    .sum()
    .sort_values("Month")
)

monthly["Month"] = monthly["Month"].dt.strftime("%Y-%m")

# Category analysis
if "Category" in data.columns:
    category_data = (
        data.groupby("Category", as_index=False)[["Sales", "Profit"]]
        .sum()
        .sort_values("Profit", ascending=False)
    )
else:
    category_data = pd.DataFrame(columns=["Category", "Sales", "Profit"])

# Region analysis
if "Region" in data.columns:
    region_data = (
        data.groupby("Region", as_index=False)["Profit"]
        .sum()
        .sort_values("Profit", ascending=False)
    )
else:
    region_data = pd.DataFrame(columns=["Region", "Profit"])

# Discount analysis
if "Discount Band" not in data.columns and "Discount" in data.columns:
    data["Discount Band"] = pd.cut(
        data["Discount"],
        bins=[-0.001, 0.05, 0.15, 0.25, 0.5, 1.0],
        labels=["0–5%", "5–15%", "15–25%", "25–50%", "50%+"]
    )

if "Discount Band" in data.columns:
    discount_data = (
        data.groupby("Discount Band", observed=False, as_index=False)["Profit"]
        .sum()
        .dropna()
    )
else:
    discount_data = pd.DataFrame(columns=["Discount Band", "Profit"])

# KPI values
total_sales = float(data["Sales"].sum())
total_profit = float(data["Profit"].sum())

profit_margin = (
    total_profit / total_sales * 100
    if total_sales != 0 else 0
)

total_orders = (
    int(data["Order ID"].nunique())
    if "Order ID" in data.columns
    else len(data)
)

# Prepare JSON data for HTML dashboard
dashboard_data = {
    "sales": total_sales,
    "profit": total_profit,
    "margin": profit_margin,
    "orders": total_orders,

    "monthly": monthly.to_dict(orient="records"),
    "category": category_data.to_dict(orient="records"),
    "region": region_data.to_dict(orient="records"),
    "discount": discount_data.to_dict(orient="records")
}

json_data = json.dumps(dashboard_data, default=str)

# ==========================================
# 2. HTML + CSS + JAVASCRIPT DASHBOARD
# ==========================================

html_template = r"""
<!DOCTYPE html>
<html lang="en">

<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>Small Business Profitability Dashboard</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap"
rel="stylesheet">

<style>

/* =================================
   GLOBAL STYLING
================================= */

* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

body {
    font-family: 'Inter', sans-serif;
    background: #0b1120;
    color: #e5e7eb;
    min-height: 100vh;
}

.container {
    max-width: 1500px;
    margin: auto;
    padding: 30px;
}

/* =================================
   HEADER
================================= */

.header {
    display: flex;
    justify-content: space-between;
    align-items: center;
    flex-wrap: wrap;
    gap: 20px;
    margin-bottom: 35px;
}

.logo {
    display: flex;
    align-items: center;
    gap: 14px;
}

.logo-icon {
    width: 50px;
    height: 50px;
    border-radius: 14px;
    background: linear-gradient(135deg, #6366f1, #06b6d4);
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 25px;
    box-shadow: 0 0 25px #6366f155;
}

.logo h1 {
    font-size: 23px;
    font-weight: 700;
}

.logo p {
    color: #94a3b8;
    font-size: 12px;
    margin-top: 5px;
}

.status {
    padding: 10px 16px;
    border-radius: 30px;
    background: #064e3b;
    color: #34d399;
    font-size: 12px;
    border: 1px solid #065f46;
}

/* =================================
   KPI CARDS
================================= */

.kpi-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 20px;
    margin-bottom: 25px;
}

.kpi-card {
    background: #111b2e;
    border: 1px solid #243249;
    padding: 25px;
    border-radius: 18px;
    position: relative;
    overflow: hidden;

    animation: fadeUp 0.7s ease both;
    transition: transform 0.3s, border-color 0.3s;
}

.kpi-card:hover {
    transform: translateY(-5px);
    border-color: #6366f1;
}

.kpi-card::before {
    content: "";
    position: absolute;
    width: 130px;
    height: 130px;
    background: #6366f1;
    opacity: 0.08;
    border-radius: 50%;
    right: -40px;
    top: -40px;
}

.kpi-title {
    color: #94a3b8;
    font-size: 13px;
    margin-bottom: 15px;
}

.kpi-value {
    font-size: 27px;
    font-weight: 800;
    color: #f8fafc;
    letter-spacing: -1px;
}

.kpi-footer {
    margin-top: 12px;
    font-size: 12px;
    color: #64748b;
}

.kpi-icon {
    font-size: 22px;
    position: absolute;
    top: 20px;
    right: 20px;
}

/* =================================
   CHART GRID
================================= */

.chart-grid {
    display: grid;
    grid-template-columns: 2fr 1fr;
    gap: 22px;
    margin-bottom: 22px;
}

.chart-card {
    background: #111b2e;
    border: 1px solid #243249;
    border-radius: 18px;
    padding: 22px;

    animation: fadeUp 0.9s ease both;
}

.chart-card h3 {
    font-size: 16px;
    font-weight: 600;
    margin-bottom: 5px;
}

.chart-card p {
    color: #94a3b8;
    font-size: 12px;
    margin-bottom: 15px;
}

.chart {
    width: 100%;
    height: 350px;
}

/* =================================
   FOOTER
================================= */

.footer {
    text-align: center;
    color: #64748b;
    font-size: 12px;
    padding: 25px 0 10px;
}

/* =================================
   ANIMATIONS
================================= */

@keyframes fadeUp {
    from {
        opacity: 0;
        transform: translateY(20px);
    }

    to {
        opacity: 1;
        transform: translateY(0);
    }
}

@keyframes glow {
    0%, 100% {
        box-shadow: 0 0 10px #6366f122;
    }

    50% {
        box-shadow: 0 0 25px #6366f144;
    }
}

.logo-icon {
    animation: glow 3s infinite;
}

/* =================================
   RESPONSIVE DESIGN
================================= */

@media(max-width: 1000px) {

    .kpi-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .chart-grid {
        grid-template-columns: 1fr;
    }
}

@media(max-width: 600px) {

    .container {
        padding: 15px;
    }

    .kpi-grid {
        grid-template-columns: 1fr;
    }

    .logo h1 {
        font-size: 18px;
    }

    .kpi-value {
        font-size: 24px;
    }

    .chart {
        height: 300px;
    }
}

</style>
</head>

<body>

<div class="container">

    <!-- HEADER -->

    <div class="header">

        <div class="logo">

            <div class="logo-icon">📊</div>

            <div>
                <h1>Profitability Analytics</h1>
                <p>Small Business Intelligence Dashboard</p>
            </div>

        </div>

        <div class="status">
            ● LIVE DATA ANALYSIS
        </div>

    </div>

    <!-- KPI CARDS -->

    <div class="kpi-grid">

        <div class="kpi-card">
            <div class="kpi-icon">💰</div>
            <div class="kpi-title">Total Sales</div>
            <div class="kpi-value" id="sales">0</div>
            <div class="kpi-footer">Overall revenue generated</div>
        </div>

        <div class="kpi-card">
            <div class="kpi-icon">📈</div>
            <div class="kpi-title">Total Profit</div>
            <div class="kpi-value" id="profit">0</div>
            <div class="kpi-footer">Net profitability</div>
        </div>

        <div class="kpi-card">
            <div class="kpi-icon">🎯</div>
            <div class="kpi-title">Profit Margin</div>
            <div class="kpi-value" id="margin">0%</div>
            <div class="kpi-footer">Profit as a percentage of sales</div>
        </div>

        <div class="kpi-card">
            <div class="kpi-icon">🛒</div>
            <div class="kpi-title">Total Orders</div>
            <div class="kpi-value" id="orders">0</div>
            <div class="kpi-footer">Unique orders / records</div>
        </div>

    </div>

    <!-- MONTHLY CHART -->

    <div class="chart-grid">

        <div class="chart-card">

            <h3>Monthly Sales & Profit</h3>
            <p>Revenue and profit performance over time</p>

            <div id="monthlyChart" class="chart"></div>

        </div>

        <div class="chart-card">

            <h3>Profit by Category</h3>
            <p>Identify your strongest product categories</p>

            <div id="categoryChart" class="chart"></div>

        </div>

    </div>

    <!-- SECOND ROW -->

    <div class="chart-grid">

        <div class="chart-card">

            <h3>Profit by Discount Band</h3>
            <p>Understand how discount levels affect profitability</p>

            <div id="discountChart" class="chart"></div>

        </div>

        <div class="chart-card">

            <h3>Profit by Region</h3>
            <p>Regional business performance</p>

            <div id="regionChart" class="chart"></div>

        </div>

    </div>

    <div class="footer">
        Small Business Profitability Analysis | Data Analytics Project
    </div>

</div>

<script>

// ========================================
// DATA FROM PYTHON DATAFRAME
// ========================================

const data = __DATA__;

const formatMoney = value =>
    new Intl.NumberFormat("en-US", {
        maximumFractionDigits: 0
    }).format(value);

// ========================================
// ANIMATED KPI COUNTERS
// ========================================

function animateCounter(id, target, prefix = "", suffix = "") {

    const element = document.getElementById(id);
    const duration = 1200;
    const start = performance.now();

    function update(now) {

        const progress = Math.min((now - start) / duration, 1);

        const eased = 1 - Math.pow(1 - progress, 3);

        const value = target * eased;

        element.textContent =
            prefix + formatMoney(value) + suffix;

        if (progress < 1) {
            requestAnimationFrame(update);
        }
    }

    requestAnimationFrame(update);
}

animateCounter("sales", data.sales, "₹");
animateCounter("profit", data.profit, "₹");
animateCounter("margin", data.margin, "", "%");
animateCounter("orders", data.orders);

// ========================================
// COMMON CHART STYLE
// ========================================

const commonLayout = {
    paper_bgcolor: "rgba(0,0,0,0)",
    plot_bgcolor: "rgba(0,0,0,0)",

    font: {
        color: "#cbd5e1",
        family: "Inter"
    },

    margin: {
        l: 45,
        r: 20,
        t: 20,
        b: 50
    },

    xaxis: {
        gridcolor: "#243249",
        zerolinecolor: "#243249"
    },

    yaxis: {
        gridcolor: "#243249",
        zerolinecolor: "#64748b"
    },

    legend: {
        orientation: "h",
        y: -0.2
    }
};

const config = {
    responsive: true,
    displayModeBar: false
};

// ========================================
// MONTHLY SALES & PROFIT
// ========================================

const months = data.monthly.map(d => d.Month);
const sales = data.monthly.map(d => d.Sales);
const profits = data.monthly.map(d => d.Profit);

const monthlyFrames = months.map((month, i) => {

    const labels = months.slice(0, i + 1);

    return {
        name: month,

        data: [
            {
                x: labels,
                y: sales.slice(0, i + 1)
            },
            {
                x: labels,
                y: profits.slice(0, i + 1)
            }
        ]
    };
});

Plotly.newPlot(
    "monthlyChart",

    [
        {
            x: months,
            y: sales,
            type: "bar",
            name: "Sales",
            marker: {
                color: "#6366f1",
                line: {
                    width: 0
                }
            },
            text: sales.map(v => formatMoney(v)),
            textposition: "outside",
            hovertemplate: "Month: %{x}<br>Sales: ₹%{y:,.2f}<extra></extra>"
        },

        {
            x: months,
            y: profits,
            type: "bar",
            name: "Profit",
            marker: {
                color: profits.map(p =>
                    p >= 0 ? "#10b981" : "#ef4444"
                )
            },
            text: profits.map(v => formatMoney(v)),
            textposition: "outside",
            hovertemplate: "Month: %{x}<br>Profit: ₹%{y:,.2f}<extra></extra>"
        }
    ],

    {
        ...commonLayout,

        barmode: "group",

        updatemenus: [
            {
                type: "buttons",
                x: 0,
                y: 1.15,

                buttons: [
                    {
                        label: "Start Animation",
                        method: "animate",
                        args: [
                            null,
                            {
                                frame: {
                                    duration: 700,
                                    redraw: true
                                },
                                transition: {
                                    duration: 300
                                },
                                fromcurrent: true
                            }
                        ]
                    },

                    {
                        label: "Stop Animation",
                        method: "animate",
                        args: [
                            [null],
                            {
                                frame: {
                                    duration: 0,
                                    redraw: false
                                },
                                mode: "immediate"
                            }
                        ]
                    }
                ]
            }
        ],

        sliders: [
            {
                currentvalue: {
                    prefix: "Month: "
                },

                steps: months.map((month, i) => ({
                    label: month,

                    method: "animate",

                    args: [
                        [month],
                        {
                            frame: {
                                duration: 500,
                                redraw: true
                            },
                            mode: "immediate"
                        }
                    ]
                }))
            }
        ]
    },

    config
).then(() => {

    Plotly.addFrames(
        "monthlyChart",
        monthlyFrames
    );

});

// ========================================
// CATEGORY PROFIT
// ========================================

Plotly.newPlot(
    "categoryChart",

    [
        {
            x: data.category.map(d => d.Profit),
            y: data.category.map(d => d.Category),

            type: "bar",
            orientation: "h",

            marker: {
                color: data.category.map(d =>
                    d.Profit >= 0 ? "#10b981" : "#ef4444"
                )
            },

            hovertemplate:
                "Category: %{y}<br>Profit: ₹%{x:,.2f}<extra></extra>"
        }
    ],

    {
        ...commonLayout,

        margin: {
            l: 100,
            r: 20,
            t: 20,
            b: 40
        }
    },

    config
);

// ========================================
// DISCOUNT BAND PROFIT
// ========================================

Plotly.newPlot(
    "discountChart",

    [
        {
            x: data.discount.map(d => d["Discount Band"]),
            y: data.discount.map(d => d.Profit),

            type: "bar",

            marker: {
                color: data.discount.map(d => d.Profit),
                colorscale: [
                    [0, "#b40426"],
                    [0.5, "#fff2a6"],
                    [1, "#006837"]
                ],
                showscale: true,
                colorbar: {
                    title: "Profit",
                    tickfont: {
                        color: "#cbd5e1"
                    }
                }
            },

            hovertemplate:
                "Discount: %{x}<br>Profit: ₹%{y:,.2f}<extra></extra>"
        }
    ],

    {
        ...commonLayout,

        showlegend: false
    },

    config
);

// ========================================
// REGIONAL PROFIT
// ========================================

Plotly.newPlot(
    "regionChart",

    [
        {
            labels: data.region.map(d => d.Region),
            values: data.region.map(d => d.Profit),

            type: "pie",
            hole: 0.55,

            marker: {
                colors: [
                    "#6366f1",
                    "#06b6d4",
                    "#10b981",
                    "#f59e0b"
                ]
            },

            textinfo: "label+percent",

            hovertemplate:
                "%{label}<br>Profit: ₹%{value:,.2f}<extra></extra>"
        }
    ],

    {
        ...commonLayout,

        showlegend: false
    },

    config
);

</script>

</body>
</html>
"""

# ==========================================
# 3. CREATE HTML FILE
# ==========================================

html_content = html_template.replace("__DATA__", json_data)

file_name = "Small_Business_Profitability_Dashboard.html"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(html_content)

print("Dashboard created successfully!")

# Display dashboard in Colab
display(HTML(html_content))

# Download HTML
files.download(file_name)

## Conclusion / presentation talking points
- Explain the objective: find the sales drivers and loss-making areas.
- Describe dataset source and the fact that it is a retail proxy dataset.
- Show the animated monthly chart and interactive filters.
- Discuss profit by category/region and discount-profit relationship.
- Close with evidence-based recommendations: review loss-making products, test discount policies, and prioritize profitable segments.

**Important:** Before submitting, verify your instructor's required dataset and business context. Replace this Kaggle dataset if your project brief requires a different small-business dataset.